In [ ]:
!pip install langchain sentence-transformers chromadb langchain-community
# Nota: Se estiver no VS Code, tire a exclamação (!) e rode no terminal.

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Texto bruto de exemplo
texto_bruto = """S
A empresa Alpha lucrou 5 milhões em 2023. A nova política de férias permite 30 dias.
O setor de vendas bateu a meta no primeiro trimestre.
"""

# 1. Otimizar a estrutura de indexação (Chunking inteligente)
# Ele vai tentar quebrar primeiro por parágrafo (\n\n), depois por linha (\n), depois por ponto.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " "]
)

textos_divididos = text_splitter.split_text(texto_bruto)

# 2. Adicionar Metadados (ajuda na filtragem antes da busca vetorial)
documentos_prontos = []
for texto in textos_divididos:
    doc = Document(
        page_content=texto,
        metadata={"fonte": "relatorio_anual", "ano": 2023}
    )
    documentos_prontos.append(doc)

print(f"Foram criados {len(documentos_prontos)} pedaços de contexto estruturado.")

In [ ]:
from sentence_transformers import CrossEncoder

# Simulando a pergunta do usuário
pergunta_usuario = "Qual foi o lucro da empresa Alpha?"

# Simulando os "Top K" documentos que o Banco Vetorial trouxe inicialmente (Recuperação)
# Note que o banco vetorial pode trazer coisas com palavras parecidas, mas contexto inútil.
documentos_recuperados = [
    "A empresa Beta teve um prejuízo em 2023.", # Ruído
    "A empresa Alpha lucrou 5 milhões em 2023.", # A resposta ideal
    "O setor de vendas da Alpha bateu a meta.", # Relevante, mas não responde o lucro
    "Lucro é um conceito contábil importante." # Ruído (palavra parecida)
]

# --- INÍCIO DA PÓS-RECUPERAÇÃO E RE-RANQUEAMENTO ---

# Carregando um modelo de re-ranqueamento (pequeno e gratuito para testes locais)
# Na prática, você baixa isso automaticamente na primeira execução
modelo_rerank = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# O modelo precisa avaliar pares: [Pergunta, Documento 1], [Pergunta, Documento 2]...
pares_para_avaliar = [[pergunta_usuario, doc] for doc in documentos_recuperados]

# Calculando a similaridade semântica exata (Score de Relevância)
notas = modelo_rerank.predict(pares_para_avaliar)

# Juntando os documentos com suas respectivas notas e organizando do maior para o menor
resultados_com_nota = list(zip(notas, documentos_recuperados))
resultados_ordenados = sorted(resultados_com_nota, key=lambda x: x[0], reverse=True)

# PÓS-RECUPERAÇÃO: Limitar os resultados (Top n)
# Vamos pegar apenas os 2 melhores textos para enviar para a LLM, evitando limite de tokens
top_n = 2
melhores_contextos = [doc for nota, doc in resultados_ordenados[:top_n]]

print("\n--- Contexto Final Refinado para enviar à LLM ---")
for i, contexto in enumerate(melhores_contextos):
    print(f"Top {i+1}: {contexto}")

In [ ]:
# Seus documentos ou textos da aula entram aqui
documentos_da_aula = [
    "O projeto da N2 vale 10 pontos e deve ser entregue até o final do semestre.",
    "Técnicas de RAG Avançado incluem pré e pós-recuperação para mitigar alucinações.",
    "O banco de dados vetorial armazena embeddings gerados a partir dos chunks."
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)

# Criando os documentos estruturados com metadados
documentos_preparados = []
for i, texto in enumerate(documentos_da_aula):
    doc = Document(
        page_content=texto,
        metadata={"id_segmento": i, "materia": "IA_Avancada"}
    )
    documentos_preparados.append(doc)

In [ ]:
modelo_rerank = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# --- CÉLULA FALTANTE: PASSO 3 - INDEXAÇÃO E RECUPERAÇÃO ---

# 1. Carregar o modelo para transformar o texto em vetores matemáticos
modelo_embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Criando o banco vetorial (Chroma) e salvando nossos documentos preparados nele
banco_vetorial = Chroma.from_documents(documentos_preparados, modelo_embedding)

# 3. Realizando a busca inicial no banco (Trazendo os Top 3 resultados)
pergunta = "Como mitigar alucinações no RAG?"
documentos_recuperados = banco_vetorial.similarity_search(pergunta, k=3)

# 4. Extraindo apenas o texto puro dos documentos que o banco encontrou
textos_recuperados = [doc.page_content for doc in documentos_recuperados]

print("Busca no banco vetorial concluída!")

# Montando os pares para o modelo analisar
pares = [[pergunta, texto] for texto in textos_recuperados]
notas = modelo_rerank.predict(pares)

# Ordenando os textos de acordo com a nota de relevância real
resultados_finais = sorted(list(zip(notas, textos_recuperados)), key=lambda x: x[0], reverse=True)

# Selecionando o Top 1 mais relevante (Top n)
contexto_perfeito = resultados_finais[0][1]
print(f"Melhor contexto encontrado após o re-ranqueamento:\n-> {contexto_perfeito}")

In [ ]:
# Formatando o prompt que a LLM usaria para responder
prompt_final = f"""
Você é um assistente de IA especializado. Responda à pergunta do usuário utilizando estritamente o contexto fornecido abaixo.

Contexto: {contexto_perfeito}
Pergunta: {pergunta}

Resposta:
"""
print("\n--- PROMPT PRONTO PARA ENVIAR À LLM ---")
print(prompt_final)